In [ ]:
import pandas as pd
import pm4py
from pm4py.algo.transformation.log_to_features import algorithm as feature_extraction
from pm4py.objects.conversion.log import converter as log_converter
from pm4py.objects.log.obj import EventLog, Trace

In [ ]:
dataset_name = 'Traffic_Fines'
dataset_path = f'../datasets/raw/{dataset_name}.xes'

log_df = pm4py.read_xes(dataset_path)
log = log_converter.apply(log_df)

print(log_df.shape)

c:\Users\Pavel\Desktop\PROJECTS\master-thesis\.venv\lib\site-packages\pm4py\utils.py:990: UserWarning: In the current version, the import/export operation uses `rustxes` by default for importing/exporting files faster. Please uninstall `rustxes` to revert the behavior.
  warnings.warn(


(561470, 16)


In [ ]:
def get_dataset_stats(log_df):
    # Case durations in days
    case_durations = log_df.groupby('case:concept:name')['time:timestamp'].agg(
        lambda x: (x.max() - x.min()).total_seconds() / 86400
    )
    mean_cycle = case_durations.mean()

    normalization_factor = case_durations.max()

    return {'mean_cycle': mean_cycle, 'normalization_factor': normalization_factor}

In [58]:
get_dataset_stats(log_df)

{'mean_cycle': 341.67626299572163, 'normalization_factor': 4372.0}

In [3]:
log_df.head()

,amount,matricola,vehicleClass,time:timestamp,notificationType,concept:name,lastSent,case:concept:name,lifecycle:transition,expense,org:resource,totalPaymentAmount,article,dismissal,points,paymentAmount
0,35.0,NaN,A,2006-07-23 22:00:00+00:00,None,Create Fine,None,A1,complete,NaN,561,0.0,157.0,NIL,0.0,NaN
1,NaN,NaN,None,2006-12-04 23:00:00+00:00,None,Send Fine,None,A1,complete,11.0,None,NaN,NaN,None,NaN,NaN
2,35.0,NaN,A,2006-08-01 22:00:00+00:00,None,Create Fine,None,A100,complete,NaN,561,0.0,157.0,NIL,0.0,NaN
3,NaN,NaN,None,2006-12-11 23:00:00+00:00,None,Send Fine,None,A100,complete,11.0,None,NaN,NaN,None,NaN,NaN
4,NaN,NaN,None,2007-01-14 23:00:00+00:00,P,Insert Fine Notification,P,A100,complete,NaN,None,NaN,NaN,None,NaN,NaN


In [72]:
def get_all_prefixes(log, min_trace_len=3):
    all_prefixes = []

    for trace in log:
        case_id = trace.attributes.get('concept:name', 'unknown')
        for i in range(min_trace_len, len(trace) + 1):
            prefix_trace = Trace(trace[:i])

            prefix_trace.attributes['case:concept:name'] = f'{case_id}_prefix_{i}'
            prefix_trace.attributes['prefix_length'] = i
            prefix_trace.attributes['original_case'] = case_id

            all_prefixes.append(prefix_trace)

    prefix_log = EventLog(all_prefixes)
    prefix_log_df = pm4py.convert_to_dataframe(prefix_log)

    return prefix_log, prefix_log_df

In [53]:
def train_test_split_log(log, log_df, train_ratio=0.8, min_trace_len=3):
    """
    Splits the log into training and testing sets based on the start time of cases.
    """
    # Filter out cases with fewer than min_events events
    case_counts = log_df.groupby('case:concept:name').size()
    valid_cases = case_counts[case_counts >= min_trace_len].index

    filtered_df = log_df[log_df['case:concept:name'].isin(valid_cases)].copy()

    filtered_df['time:timestamp'] = pd.to_datetime(filtered_df['time:timestamp'])

    case_starts = (
        filtered_df.groupby('case:concept:name')['time:timestamp']
        .min()
        .sort_values()
        .index.tolist()
    )

    split_idx = int(len(case_starts) * train_ratio)
    train_ids = set(case_starts[:split_idx])
    test_ids = set(case_starts[split_idx:])

    train_df = filtered_df[filtered_df['case:concept:name'].isin(train_ids)]
    test_df = filtered_df[filtered_df['case:concept:name'].isin(test_ids)]

    train_log = pm4py.convert_to_event_log(train_df)
    test_log = pm4py.convert_to_event_log(test_df)

    return train_log, test_log, train_df, test_df

In [54]:
train_ratio = 0.8
min_trace_len = 3

train_log, test_log, train_log_df, test_log_df = train_test_split_log(
    log, log_df, train_ratio=train_ratio, min_trace_len=min_trace_len
)

print(f'Original: {len(log)}')
print(f'Filtered out: {len(log) - len(train_log) - len(test_log)}')
print(f'Training: {len(train_log)}, Testing: {len(test_log)}')

Original: 150370
Filtered out: 66756
Training: 66891, Testing: 16723


In [73]:
train_prefix_log, train_prefix_df = get_all_prefixes(
    train_log, min_trace_len=min_trace_len
)
print(f'Generated {len(train_prefix_log)} prefixes from {len(train_log)} cases')

Generated 207746 prefixes from 66891 cases


In [ ]:
pm4py.extract_outcome_enriched_dataframe(

)

In [19]:
data, features = feature_extraction.apply(
    log, variant=feature_extraction.Variants.TRACE_BASED
)

In [ ]:
log_df.to_csv(
    f'../datasets/csv/{dataset_name}.csv', index=False
)

In [22]:
log_df.head()

,amount,matricola,vehicleClass,time:timestamp,notificationType,concept:name,lastSent,case:concept:name,lifecycle:transition,expense,org:resource,totalPaymentAmount,article,dismissal,points,paymentAmount
0,35.0,NaN,A,2006-07-23 22:00:00+00:00,None,Create Fine,None,A1,complete,NaN,561,0.0,157.0,NIL,0.0,NaN
1,NaN,NaN,None,2006-12-04 23:00:00+00:00,None,Send Fine,None,A1,complete,11.0,None,NaN,NaN,None,NaN,NaN
2,35.0,NaN,A,2006-08-01 22:00:00+00:00,None,Create Fine,None,A100,complete,NaN,561,0.0,157.0,NIL,0.0,NaN
3,NaN,NaN,None,2006-12-11 23:00:00+00:00,None,Send Fine,None,A100,complete,11.0,None,NaN,NaN,None,NaN,NaN
4,NaN,NaN,None,2007-01-14 23:00:00+00:00,P,Insert Fine Notification,P,A100,complete,NaN,None,NaN,NaN,None,NaN,NaN


[Outcome-Oriented Predictive Process Monitoring: Review and Benchmark](https://dl.acm.org/doi/pdf/10.1145/3301300)

Datasets:

- BPIC15_1
- BPIC15_2
- BPIC15_3
- BPIC15_4
- BPIC15_5
- Sepsis
- Traffic_Fines

Data Encodings:

- Static encoding
- Last state encoding
- Aggregation encoding
- Index-based encoding

Bucketing strategies:

- Single Bucket
- KNN
- Clustering
- Prefix Length

Methods:

- XGBoost
- RF
- Logistic Regression
- SVM


Input features can be: activity, timestamp, resource and attributes

[Features Transformers](https://github.com/nirdizati/nirdizati-training-backend/tree/master/transformers)